# 01 — Data Loading and EDA

**Goal**: pull price + news data, inspect quality, and build intuition for what we're working with.

This notebook runs in demo mode by default. Switch `DEMO = False` to pull real data (requires `FINNHUB_API_KEY` in `.env`).

## What you'll learn
- How to structure a long-format panel dataset (ticker × date)
- Why we compute abnormal returns, not raw returns
- How to sanity-check financial data (splits, gaps, outliers)


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import matplotlib.pyplot as plt
from src import data_loader, config

DEMO = True

## Load prices

In [ ]:
prices = data_loader.load_prices(demo=DEMO)
print(f"Rows: {len(prices):,}")
print(f"Tickers: {prices['ticker'].nunique()}")
print(f"Date range: {prices['date'].min().date()} to {prices['date'].max().date()}")
prices.head()

### Quick return diagnostics

Daily returns should be roughly Normal with stdev around 1-2%. Anything wildly off suggests a split/dividend adjustment issue.

In [ ]:
ret_stats = prices.groupby('ticker')['return'].agg(['mean', 'std', 'min', 'max']).round(4)
ret_stats.head(10)

In [ ]:
# Plot a sample ticker
sample = prices[prices['ticker'] == 'AAPL']
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(sample['date'], sample['adj_close'])
axes[0].set_title('AAPL Adjusted Close')
axes[1].hist(sample['return'].dropna(), bins=60)
axes[1].set_title('AAPL Daily Returns')
plt.tight_layout()

## Load news

In [ ]:
news = data_loader.load_news(demo=DEMO)
print(f"Articles: {len(news):,}")
news.head()

In [ ]:
# How many articles per ticker?
article_counts = news['ticker'].value_counts()
fig, ax = plt.subplots(figsize=(12, 5))
article_counts.head(30).plot(kind='bar', ax=ax)
ax.set_title('Article count by ticker')
ax.set_ylabel('Articles')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()

In [ ]:
# Source tier distribution
news['source_tier'].value_counts()

## Save to processed/

In [ ]:
prices.to_parquet(config.PROCESSED_DIR / 'prices.parquet', index=False)
news.to_parquet(config.PROCESSED_DIR / 'news_raw.parquet', index=False)
print('Saved prices and news to data/processed/')

## ✅ Reflection

- Does every ticker have the expected number of trading days?
- Are returns roughly symmetric around zero?
- Is news volume balanced across tickers, or do a few dominate?

Answer these in your notebook markdown cells as part of your project hygiene. Recruiters who skim your notebooks will notice.

**Next**: `02_sentiment_scoring.ipynb`